# Post-processing — proprietary API LLM data (risk tasks)

API analog of `04_preprocess_llm_data_risk.ipynb`, for the proprietary models
collected via the native-API runner.

**What differs from script 04 (open-weights):**
- The API runner produced **one version only**: `no_context`, **non-flipped**,
  **no reasoning** (`_nr`). So there is no flipped / re-flipped branch here.
- The APIs return **no logits/probs**, so `add_argmax_column` is skipped and
  `logit_score` / `prob_score` are set to `None`, `has_logits = False`
  (same shims as `06_preprocess_llm_api_data_ipip_neo_300.py`).
- Only `model_answer` is recoded (the open-weights notebook also recoded the
  now-absent `logit_score` / `prob_score`).

All per-task category / reverse-coding / answer-translation maps are copied
**verbatim** from script 04 so the two pipelines stay aligned.

**Outputs** (mirroring the IPIP `api_data/` convention):
- `data/intermediate/risk_data/api_data/LLM_api_no_flip_data_raw.csv`
- `data/intermediate/risk_data/api_data/LLM_api_no_flip_data_rereversed.csv`


## Packages & config

In [ ]:
import pandas as pd
import numpy as np
import glob, os, shutil
from utils import load_dataframes

# Raw per-task API CSVs for the analysis pipeline (mirrors the IPIP-NEO
# convention: data/raw/ipipneo300_data/outputs_api).
RAW_API_DIR = "../../data/raw/risk_data/outputs_api"
OUT_DIR     = "../../data/intermediate/risk_data/api_data"
os.makedirs(RAW_API_DIR, exist_ok=True)
os.makedirs(OUT_DIR, exist_ok=True)

RISK_TASKS = ["barratt", "care", "dast", "dm", "dospert", "gabs",
              "pri", "soep", "sssv", "lot", "dfd", "mpl"]

# Accumulators -> the two output CSVs.
raw_frames = []         # LLM_api_no_flip_data_raw.csv  (responses as given)
rereversed_frames = []  # LLM_api_no_flip_data_rereversed.csv  (reverse items flipped back)

## BARRATT — 1–4 Likert, subscales BISa/BISm/BISn, reverse-coded

In [ ]:
BARRATT = load_dataframes("barratt", RAW_API_DIR)
BARRATT["experiment"] = "BARRATT"

# Drop the 4 concept-test items that are not part of the BARRATT scale.
BARRATT = BARRATT[BARRATT["item_id"] <= 30].copy()

item_to_category = {
    1: "BISn", 2: "BISm", 3: "BISm",  4: "BISm", 5: "BISa",  6: "BISa",  7: "BISn",  8: "BISn",  9: "BISa",  10: "BISn",
    11: "BISa", 12: "BISn", 13: "BISn",  14: "BISn", 15: "BISn",  16: "BISm",  17: "BISm",  18: "BISn",  19: "BISm",  20: "BISa",
    21: "BISm", 22: "BISm", 23: "BISm",  24: "BISa", 25: "BISm",  26: "BISa",  27: "BISn",  28: "BISa",  29: "BISn",  30: "BISm"
}
BARRATT["category"] = BARRATT["item_id"].map(item_to_category)

reverse_coded = {
    1: True, 2: False, 3: False,  4: False, 5: False,  6: False,  7: True,  8: True,  9: True,  10: True,
    11: False, 12: True, 13: True,  14: False, 15: True,  16: False,  17: False,  18: False,  19: False,  20: True,
    21: False, 22: False, 23: False,  24: False, 25: False,  26: False,  27: False,  28: False,  29: True,  30: True
    }
BARRATT["reverse_coded"] = BARRATT["item_id"].map(reverse_coded)

raw_frames.append(BARRATT.copy())

bar_rev = BARRATT.copy()
mask = bar_rev["reverse_coded"] == True
bar_rev.loc[mask, "model_answer"] = 5 - bar_rev.loc[mask, "model_answer"]
rereversed_frames.append(bar_rev)

## CARE — 0–100, subscales CAREa/CAREs/CAREw, no reverse items

In [ ]:
CARE = load_dataframes("care", RAW_API_DIR)
CARE["experiment"] = "CARE"

item_to_category = {
    1: "CAREa", 2: "CAREa", 3: "CAREa",  4: "CAREa", 5: "CAREa",  6: "CAREa",  7: "CAREa",  8: "CAREa",  9: "CAREa",  10: "CAREs",
    11: "CAREs", 12: "CAREs", 13: "CAREs",  14: "CAREs", 15: "CAREs",  16: "CAREw",  17: "CAREw",  18: "CAREw",  19: "CAREw"
}
CARE["category"] = CARE["item_id"].map(item_to_category)

# No reverse-coded items: raw == rereversed.
raw_frames.append(CARE.copy())
rereversed_frames.append(CARE.copy())

## DAST — binary 1/2 recoded to 1/0 (per-item key)

`raw` keeps the as-given 1/2; `rereversed` applies the Frey scoring key.

In [ ]:
DAST = load_dataframes("dast", RAW_API_DIR)
DAST["experiment"] = "DAST"

raw_frames.append(DAST.copy())

dast_maps = {
    1: {1: 1, 2: 0},                           
    2: {1: 1, 2: 0},
    3: {1: 1, 2: 0},
    4: {1: 0, 2: 1},
    5: {1: 0, 2: 1},
    6: {1: 1, 2: 0},
    7: {1: 1, 2: 0},
    8: {1: 1, 2: 0},
    9: {1: 1, 2: 0},
    10: {1: 1, 2: 0},
    11: {1: 1, 2: 0},
    12: {1: 1, 2: 0},
    13: {1: 1, 2: 0},
    14: {1: 1, 2: 0},
    15: {1: 1, 2: 0},
    16: {1: 1, 2: 0},
    17: {1: 1, 2: 0},
    18: {1: 1, 2: 0},
    19: {1: 1, 2: 0},
    20: {1: 1, 2: 0}
}

def recode_dast_value(item_id, value):
    mapping = dast_maps.get(item_id)
    if mapping is not None:
        return mapping.get(value, np.nan)
    return value

dast_rev = DAST.copy()
dast_rev["model_answer"] = dast_rev.apply(
    lambda r: recode_dast_value(r["item_id"], r["model_answer"]), axis=1
)
rereversed_frames.append(dast_rev)

## DM — 1–4 mapped to a 0–2 point score (`{4:1, 3:2, 2:1, 1:0}`)

`raw` keeps 1–4; `rereversed` applies the point mapping (one step further than
the human pipeline, directly to summable scores).

In [ ]:
DM = load_dataframes("dm", RAW_API_DIR)
DM["experiment"] = "DM"

raw_frames.append(DM.copy())

mapping = {
    4: 1,
    3: 2,
    2: 1,
    1: 0
} # here, I already go one step further than human process, directly to real scores that are then summed/averaged for final sum

dm_rev = DM.copy()
dm_rev["model_answer"] = dm_rev["model_answer"].map(mapping)
rereversed_frames.append(dm_rev)

## DOSPERT — 1–5, 6 domain subscales, no reverse items

In [ ]:
DOSPERT = load_dataframes("dospert", RAW_API_DIR)
DOSPERT["experiment"] = "DOSPERT"

item_to_category = {
    1: "Social", 10: "Social", 16: "Social", 19: "Social", 23: "Social", 26: "Social", 34: "Social", 35: "Social",
    2: "Recreational", 6: "Recreational", 15: "Recreational", 17: "Recreational", 21: "Recreational", 31: "Recreational", 37: "Recreational", 38: "Recreational",
    3: "Gambling", 11: "Gambling", 22: "Gambling", 33: "Gambling",
    4: "Health", 8: "Health", 27: "Health", 29: "Health", 32: "Health", 36: "Health", 39: "Health", 40: "Health",
    5: "Ethical", 9: "Ethical", 12: "Ethical", 13: "Ethical", 14: "Ethical", 20: "Ethical", 25: "Ethical", 28: "Ethical",
    7: "Investment", 18: "Investment", 24: "Investment", 30: "Investment"
}
DOSPERT["category"] = DOSPERT["item_id"].map(item_to_category)

raw_frames.append(DOSPERT.copy())
rereversed_frames.append(DOSPERT.copy())

## GABS — mixed scale (item 0: 1–2, items 1–15: 1–4), no reverse items on non-flipped

In [ ]:
GABS = load_dataframes("gabs", RAW_API_DIR)
GABS["experiment"] = "GABS"

raw_frames.append(GABS.copy())
rereversed_frames.append(GABS.copy())

## PRI — binary, no reverse items

In [ ]:
PRI = load_dataframes("pri", RAW_API_DIR)
PRI["experiment"] = "PRI"

raw_frames.append(PRI.copy())
rereversed_frames.append(PRI.copy())

## SOEP — 1–11, per-domain subscales, no reverse items

In [ ]:
SOEP = load_dataframes("soep", RAW_API_DIR)
SOEP["experiment"] = "SOEP"

item_to_category = {
    1: "SOEP", 2: "SOEPdri", 3: "SOEPfin",  4: "SOEPrec", 5: "SOEPocc",  6: "SOEPhea",  7: "SOEPsoc",
}
SOEP["category"] = SOEP["item_id"].map(item_to_category)

raw_frames.append(SOEP.copy())
rereversed_frames.append(SOEP.copy())

## SSSV — binary 1/2, 4 subscales, reverse-coded (reversal = 3 − x)

In [ ]:
SSSV = load_dataframes("sssv", RAW_API_DIR)
SSSV["experiment"] = "SSSV"

item_to_category = {
    3: "SStas", 11: "SStas", 16: "SStas", 17: "SStas", 20: "SStas", 21: "SStas", 23: "SStas", 28: "SStas", 38: "SStas", 40: "SStas",
    4: "SSexp", 6: "SSexp", 9: "SSexp", 10: "SSexp", 14: "SSexp", 18: "SSexp", 19: "SSexp", 22: "SSexp", 26: "SSexp", 37: "SSexp",
    1: "SSdis", 12: "SSdis", 13: "SSdis", 25: "SSdis", 29: "SSdis", 30: "SSdis", 32: "SSdis", 33: "SSdis", 35: "SSdis", 36: "SSdis",
    2: "SSbor", 5: "SSbor", 7: "SSbor", 8: "SSbor", 15: "SSbor", 24: "SSbor", 27: "SSbor", 31: "SSbor", 34: "SSbor", 39: "SSbor",
}
SSSV["category"] = SSSV["item_id"].map(item_to_category)

reverse_coded = {
     1: True, 2: False, 3: True, 4: False, 5: True, 6: True, 7: False, 8: True, 9: True, 10: False, 
     11: False, 12: False, 13: False, 14: True, 15: False, 16: True, 17: True, 18: True, 19: False, 20: False,
     21: False, 22: True, 23: True, 24: True, 25: False, 26: False, 27: False, 28: True, 29: True, 30: False,
     31: False, 32: True, 33: False, 34: True, 35: False, 36: True, 37: False, 38: False, 39: True, 40: False

}
SSSV["reverse_coded"] = SSSV["item_id"].map(reverse_coded)

raw_frames.append(SSSV.copy())

sssv_rev = SSSV.copy()
mask = sssv_rev["reverse_coded"] == True
sssv_rev.loc[mask, "model_answer"] = 3 - sssv_rev.loc[mask, "model_answer"]
rereversed_frames.append(sssv_rev)

## LOT — string choice s1/s2 → 1/0 (risky=1), reverse-coded (reversal = 1 − x)

`s1` = risky = 1, `s2` = 0. Then reverse-coded items are flipped back.

In [ ]:
LOT = load_dataframes("lot", RAW_API_DIR)
LOT["experiment"] = "LOT"
LOT = LOT.rename(columns={"model_answer_option": "model_answer"})

# Non-numeric answer scale -> 0/1 (unmapped values, e.g. parse errors, -> NaN).
LOT["model_answer"] = LOT["model_answer"].map({"s1": 1, "s2": 0})

# Which option is the RISKY one. Convention: True => risky is s2, so the raw
# s1->1 mapping gets flipped (1 - x); False => risky is s1 (no flip).
# Ground truth from Frey table S8 + lotteries.csv choices (R vs Decision_X over
# all participants): risky = Option B (s2) for items 1-15, Option A (s1) for
# items 16-25. NB: the human `Presentation_XZ` order is randomised ~50/50 per
# participant and must NOT be used to set this.
# Regression check: src/preprocessing/check_lot_risky_coding.py
risky_decision_table = {
    1: True,  2: True,  3: True,  4: True,  5: True,  6: True,  7: True,  8: True,  9: True,  10: True,
    11: True, 12: True, 13: True, 14: True, 15: True, 16: False, 17: False, 18: False, 19: False, 20: False,
    21: False, 22: False, 23: False, 24: False, 25: False
}
LOT["reverse_coded"] = LOT["item_id"].map(risky_decision_table)

raw_frames.append(LOT.copy())

lot_rev = LOT.copy()
mask = lot_rev["reverse_coded"] == True
lot_rev.loc[mask, "model_answer"] = 1 - lot_rev.loc[mask, "model_answer"]
rereversed_frames.append(lot_rev)

## DFD — string choice s1/s2 → 1/0 (risky=1), reverse-coded (reversal = 1 − x)

`gamble_id` → `item_id`; `s1` = risky = 1, `s2` = 0.

In [ ]:
DFD = load_dataframes("dfd", RAW_API_DIR)
DFD["experiment"] = "DFD"
DFD = DFD.rename(columns={"model_answer_option": "model_answer", "gamble_id": "item_id"})

DFD["model_answer"] = DFD["model_answer"].map({"s1": 1, "s2": 0})

risky_decision_table = {
    "he04_1": False,
    "he04_2": False,
    "he04_3": True,
    "he04_4": True,
    "he04_5": False,
    "he04_6": False,
    "he04_2inv": True,
    "he04_6inv": True
}

DFD["reverse_coded"] = DFD["item_id"].map(risky_decision_table)

raw_frames.append(DFD.copy())

dfd_rev = DFD.copy()
mask = dfd_rev["reverse_coded"] == True
dfd_rev.loc[mask, "model_answer"] = 1 - dfd_rev.loc[mask, "model_answer"]
rereversed_frames.append(dfd_rev)

## MPL — string choice s1/s2 → 0/1 (risky=second option), no reverse items

`gamble_id` → `item_id`; `s1` = 0, `s2` = risky = 1.

In [ ]:
MPL = load_dataframes("mpl", RAW_API_DIR)
MPL["experiment"] = "MPL"
MPL = MPL.rename(columns={"model_answer_option": "model_answer", "gamble_id": "item_id"})

MPL["model_answer"] = MPL["model_answer"].map({"s1": 0, "s2": 1})

raw_frames.append(MPL.copy())
rereversed_frames.append(MPL.copy())

## Combine, apply API shims, and save

In [ ]:
raw = pd.concat(raw_frames, ignore_index=True)
rer = pd.concat(rereversed_frames, ignore_index=True)

# API shims (parallel to 06_preprocess_llm_api_data_ipip_neo_300.py): the
# proprietary APIs return no token logits/probs, and everything is no_context.
for d in (raw, rer):
    d["logit_score"] = None
    d["prob_score"]  = None
    d["has_logits"]  = False
    d["context_mode"] = "no_context"

raw.to_csv(os.path.join(OUT_DIR, "LLM_api_no_flip_data_raw.csv"), index=False)
rer.to_csv(os.path.join(OUT_DIR, "LLM_api_no_flip_data_rereversed.csv"), index=False)

print("raw       :", raw.shape)
print("rereversed:", rer.shape)
print("models     :", sorted(raw["model"].unique()))
print("experiments:", sorted(raw["experiment"].unique()))
# Per-(experiment, model) item counts — quick check for missing/partial tasks.
print(raw.groupby(["experiment", "model"]).size().unstack(fill_value=0).T)